# Vanguard_AB 

# 1. First Exploration of web_data Datasets Part 1 and 2

**Safe Import, Comparison/Verification**
Two original files on "Digital Footprints" (User website interactions): 
Part1 (343143, 5) & Part2 (412266, 5)
Both have columns (client_id,visitor_id,visit_id,process_step,date_time) 
--> Check if identical before merging

**Caveat**
A common trap in A/B testing data is Encoding or Data Type mismatches 
(like ID var being Int in one table and Str elsewhere, or repeated user actions, technical errors and bugs etc.)

## Data Sources

(1)     Client Profiles -- User demographics (df_final_demo.txt)
(70610, 9: client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth) 
(2)     Digital Footprints -- User website interactions 
2.1 Part1 (df_final_web_data_pt_1.txt)
(343142, 5: client_id,visitor_id,visit_id,process_step,date_time) 
2.2 Part2 (df_final_web_data_pt_2.txt)
(412265, 5: client_id,visitor_id,visit_id,process_step,date_time) 
(3)     Experiment Roster -- Control or Test Group (df_final_experiment_clients.txt)
(70610, 2: client_id, Variation) 

## Load & Merge ds 2.1 and 2.2 (Footprints)

- Merge 2.1 & 2.2 into one df df_final_web_data 
- Cleanup datetime col and save merged df as Parquet and csv

In [ ]:
# 1. First Look at web_data in Pandas
# Compare both txt tables and check for swapped columns or different param.

import pandas as pd

# Peek at top rows using nrows=5
path1 = 'df_final_web_data_pt_1.txt' # ADD PATHNAME FROM REPO - TODO
path2 = 'df_final_web_data_pt_2.txt'

df1_peek = pd.read_csv(path1, sep=',', nrows=5) 
df2_peek = pd.read_csv(path2, sep=',', nrows=5)

print(df1_peek.columns)
print(df2_peek.columns)

# Check if columns match in terms of name and order
print(f"Columns match: {list(df1_peek.columns) == list(df2_peek.columns)}")

## Load and concatenate txt files 

- Needs ca. 200-500MB of RAM to load both in Pandas (750K+ lines)
- Execute in a single notebook and refresh before, don't keep many copies in memory

In [ ]:
# 2. Load both files as dfs 

df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

# Vertical Concat into one main df_web_data
df_web_data = pd.concat([df1, df2], axis=0, ignore_index=True)  

# Clean memory -- Delete the two partial dfs right away to prevent overload
del df1
del df2

## Verification and Parquet Export 

Check logic and columns match immediately after merging

- Dtypes - set date_time to Datetime if imported as string --> df_web_data.info() 

- Duplicates - AB test logs have LOTS of duplicates (like refreshed page) --> df_web_data.duplicated().sum()

- Timeline - get min/max of the date_time column, is there a time gap between parts 1 & 2, or doubled dates/overlaps?

In [ ]:
# 3. Column Verification

df_web_data.info()
df_web_data.duplicated().sum()

# Convert date_time to datetime objects
df_web_data['date_time'] = pd.to_datetime(df_web_data['date_time'])
# Check duplicates & timeline consistency TODO

# 4. Store as compressed Parquet file to Preserve Dtypes/Formatting 

# Use Pandas method df.to_parquet() to save as Parquet
df_web_data.to_parquet('merged_web_data.parquet')

## NEXT STEPS

# Merge the Client Profiles (final_demo) and Roster (experiment_clients) Datasets

- Merge 1 & 3 into one DE-anonymized Client profiles list 
(Connect by client_ID, Join the 9 demo columns, so just adding col "VARIATION" as an additional column)
- Use L's updated CSV from data>clean, don't need local parquet from here - TODO

In [ ]:
# 1. Load the demographic(1) and roster(3) files (assuming they are cleaned)

df_demo = pd.read_csv('df_final_demo.csv')
df_roster = pd.read_csv('df_final_experiment_clients.csv')

# 2. Left Join Roster onto Demo (70610 rows)
# This keeps all clients and adds their 'Variation' (Test/Control/NaN)
df_clients = pd.merge(df_demo, df_roster, on='client_id', how='left')

# 3. Handle the 30% NaNs in Variation
df_clients['Variation'] = df_clients['Variation'].fillna('Unknown')

## --> L MADE MORE PROGRESS HERE SEE BRANCH

# NEXT STEPS ON MAIN - SORT/SEE TRELLO

# Integrate Web Data
Warning: Only merge raw web_data for EVENT level analysis.
For client-level analysis, aggregate web data first 
(e.g., total clicks) -- TODO 

df_final_master = pd.merge(df_clients, df_web_data, on='client_id', how='left')

--> Cleanup 1 and 3 - DONE L & AD 
--> Merge 1 & 3 into one DE-anonymized Client profiles list - DONE
(Connect by client_ID, Join 9 ClientProfiles columns (or drop some?), PLUS the A or B check col Testgroup Y/N)
--> Store Merged DE-Anon Client Profiles in pandas df and make csv & parquet files - DONE A

Upload to local mySQL Database if needed to do joins, filter queries etc. 
DON'T push merged web_data csv or parquet file to Github -- use locally and add to gitignore

JOIN to other two datasets via client_id 

## First Analysis of A-B Groups
Simple Demographics (eg, Avg age, gender, location, tenure, site visits?)
Common sense checks (Differences between Test group A and Control B? Skews and Bias? Sample sizes comparable? Variances similar?) - TODO
--> Make ER diagram of the db relations (with client_id as connector) - TODO
--> Keep (2) and (1&3) as distinct tables for now -- no need to merge them yet! Use client_id as connecting Key